# Notebook 12: File Context y Persistencia

## Introducción

Hasta ahora usamos contexto efímero. Ahora aprenderás a persistir configuraciones en disco para producción.

### Objetivos:
1. Crear File Context
2. Persistir suites y configuraciones
3. Versionar expectativas
4. Compartir configuraciones con el equipo

In [ ]:
import great_expectations as gx
import pandas as pd
from pathlib import Path

# Crear directorio para GX
gx_dir = Path("../gx_project")
gx_dir.mkdir(exist_ok=True)

print(f" Directorio GX: {gx_dir}")

## Crear File Context

In [ ]:
# Crear contexto con persistencia
context = gx.get_context(mode="file", project_root_dir=str(gx_dir))

print(" File Context creado")
print(f"\nArchivos creados:")
for item in gx_dir.rglob("*"):
    if item.is_file():
        print(f"  - {item.relative_to(gx_dir)}")

## Configurar y Persistir

In [ ]:
# Configurar datasource
datasource = context.data_sources.add_pandas(name="ventas_prod")
asset = datasource.add_dataframe_asset(name="ventas")
batch_def = asset.add_batch_definition_whole_dataframe("batch_completo")

# Crear suite
suite = context.suites.add(gx.ExpectationSuite(name="suite_produccion"))

suite.add_expectation(
    gx.expectations.ExpectColumnValuesToNotBeNull(column="order_id")
)
suite.add_expectation(
    gx.expectations.ExpectColumnValuesToBeBetween(
        column="price", min_value=0.01, max_value=10000
    )
)

suite.save()
print(" Configuración persistida en disco")

## Cargar Configuración Existente

In [ ]:
# En otro script/notebook, puedes cargar la configuración
context_reload = gx.get_context(mode="file", project_root_dir=str(gx_dir))

# Listar suites disponibles
print("Suites disponibles:")
for suite in context_reload.suites.all():
    print(f"  - {suite.name}")

# Cargar suite
suite_loaded = context_reload.suites.get("suite_produccion")
print(f"\n Suite cargada con {len(suite_loaded.expectations)} expectativas")

## Validar con Configuración Persistida

In [ ]:
df = pd.read_csv("../data/ventas_sucias.csv")

val_def = context_reload.validation_definitions.add(
    gx.ValidationDefinition(
        data=batch_def,
        suite=suite_loaded,
        name="validacion_prod"
    )
)

resultado = val_def.run(batch_parameters={"dataframe": df})
print(f"\n¿Validación exitosa?: {' SÍ' if resultado.success else ' NO'}")

##  Ejercicio

Crea una segunda suite persistida y valídala.

In [ ]:
# TU CÓDIGO AQUÍ
pass

In [ ]:
context.build_data_docs()
context.open_data_docs()

##  Resumen

1.  File Context persiste configuraciones
2.  Suites se guardan en disco
3.  Configuraciones son versionables (Git)
4.  Equipo puede compartir configuraciones

